In [4]:
# Imports
import os
import datetime

import pandas as pd
import numpy as np
from pyproj import CRS, Transformer
from PyAstronomy import pyasl

In [5]:
# Load KOPRI GNSS into DFs
file_path = '../_data/GNSS'
gnss_records = []
for file in os.listdir(file_path):
    if file.endswith('.txt'):
        gnss_data = pd.read_csv(f'{file_path}/{file}',header=0, names=['time','x','y','z','detrended_z','v'])
        gnss_data['datetime'] = [pyasl.decimalYearGregorianDate(time) for time in gnss_data['time']]
        x0 = gnss_data['x'].iloc[0]
        y0 = gnss_data['y'].iloc[0]

        gnss_data['h_displacement'] = np.sqrt(
            (gnss_data['x'] - x0)**2 +
            (gnss_data['y'] - y0)**2
        )
        station = file.split('.')[0]
        gnss_data['station'] = station
        gnss_records.append(gnss_data)


In [10]:
ordered_dirs = sorted(gnss_records, key=lambda x: x['station'][0])
for dir in ordered_dirs:
    print(dir['time'][0])

2015.0072
2015.00766
2015.0072
2016.00833
2015.98643
2015.98643
2015.98507
2015.98507
2015.98643
2017.88531
2018.88034


In [23]:
stano = 0
for sta in ordered_dirs:
    stano = stano + 1
    print(sta['station'][0]),
    # Make empty dataframe for each station to be filled in with starts/ends
    out = pd.DataFrame(columns=["start", "end", "station", "stano"])
    stream = sta
    start_time = stream["datetime"][0]
    for i in range(len(stream["datetime"])):
        if i > 0:
            prior_time = stream["datetime"][i - 1]
            current_time = stream["datetime"][i]

            delta_t = current_time - prior_time
            if delta_t > datetime.timedelta(days=1, minutes=-1):  # Check if > 1 day
                end_time = prior_time
                # print(start_time)
                # print(end_time)
                out.loc[len(out.index)] = [
                    start_time,
                    end_time,
                    sta["station"][0],
                    stano,
                ]
                start_time = current_time

            if i == len(stream["datetime"]) - 1:
                end_time = current_time

                out.loc[len(out.index)] = [
                    start_time,
                    end_time,
                    sta["station"][0],
                    stano,
                ]

    # print(out)

    out["start"] = out["start"].dt.strftime("%Y-%m-%dT%H:%M:%S.%f")
    out["end"] = out["end"].dt.strftime("%Y-%m-%dT%H:%M:%S.%f")
    out.to_csv(
        "sta_uptime_input.txt",
        sep="\t",
        index=False,
        mode="a",
        header=not "sta_uptime_input.txt",
    )

    with open("sta_uptime_input.txt", "a") as file:
        file.write("\n")

KGPS06
KGPS07
KGPS08
KGPS11
KGPS13
KGPS15
KGPS16
KGPS17
KGPS18
KGPS25
KGPS32


In [13]:
# Load File and get true times with no data

times = pd.read_csv("sta_uptime_input.txt", sep="\t", header=None)
print(times)

                              0                           1     2   3
0    2010-01-15T00:40:00.000000  2010-05-02T15:59:00.000000  la01   1
1    2010-11-09T20:18:45.000000  2011-05-19T22:59:30.000000  la01   1
2    2011-11-15T06:44:45.000000  2012-05-16T19:56:45.000000  la01   1
3    2012-09-08T23:02:30.000000  2012-09-09T01:45:00.000000  la01   1
4    2012-09-10T20:02:15.000000  2012-09-10T22:08:00.000000  la01   1
..                          ...                         ...   ...  ..
968  2016-09-03T09:20:45.000000  2016-09-03T21:25:00.000000  mg04  45
969  2016-09-05T21:14:30.000000  2016-11-26T20:57:00.000000  mg04  45
970  2014-01-24T02:34:00.000000  2014-12-17T23:59:45.000000  mg05  46
971  2014-01-23T22:39:00.000000  2014-12-18T03:15:30.000000  mg06  47
972  2014-01-27T01:18:45.000000  2014-12-18T23:59:45.000000  mg07  48

[973 rows x 4 columns]
